# Time Weighted Vector Store Retriever — 시간 가중치 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **TimeWeightedVectorStoreRetriever**의 시간 가중치 계산 방식 이해하기
- `decay_rate`로 시간 감소율을 조절하는 방법 익히기
- 뉴스, 대화 기록 등 시간에 민감한 데이터에 적용하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- 최신 정보와 오래된 정보의 관련성이 다를 수 있다는 도메인 지식

---

## 핵심 개념

### 일반 Vector Search vs Time Weighted Search

| 방식 | 점수 계산 |
|------|----------|
| 일반 검색 | 유사도만 고려 |
| **Time Weighted** | 유사도 × 시간 가중치 |

예시:
```
문서 A (1년 전, 유사도 0.9)  → 시간 감소 후: 0.6  (선택 안 됨)
문서 B (어제,   유사도 0.8)  → 시간 가중치: 0.9  (선택됨)
```

## 사용 시나리오

- 뉴스 피드 검색
- 실시간 정보 업데이트
- 대화 기록 기반 검색
- 시간에 민감한 금융·주식 데이터

> **실무 팁**: `decay_rate`를 아주 작은 값(예: 1e-24)으로 설정하면 사실상 시간 순서만 반영돼요. 반대로 0.9처럼 크게 설정하면 어제 데이터도 크게 감소합니다.

In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------------------------------------------
# 시간 메타데이터가 다른 문서를 추가하고 최신 문서 우선 검색 확인
# ---------------------------------------------------

# ============================================================
# TODO: TimeWeightedVectorStoreRetriever를 생성하고 시간차를 둔 문서를 추가하세요
# 힌트: decay_rate가 작을수록 시간 감소 속도가 느림 (최신 문서 우대 효과 약함)
# 예상 결과: 검색 결과에서 최신 문서가 상위에 위치
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime, timedelta
import time

# Vectorstore 준비

# Vectorstore 준비
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(
    ["빈 초기화용"],
    embedding=embeddings
)

# TimeWeightedVectorStoreRetriever 생성
# decay_rate=0.01: 반감기 약 70일, 8시간이면 시간 점수 약 0.92
# 최종 점수 = 유사도 + (1 - decay_rate)^경과시간
time_weighted_retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=3
)

# 시간차를 두고 문서 추가 (last_accessed_at으로 최신성 지정)
# :bulb: 비슷한 주제의 풍부한 문서를 사용하면 유사도가 균등해져서
#    시간 가중치의 효과를 명확하게 관찰할 수 있음
docs_with_time = [
    ("2020년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 OpenAI의 GPT-3 발표였습니다. "
    "1750억 파라미터의 대규모 언어 모델이 Few-shot 학습으로 번역, 요약, 코드 생성 등 "
    "다양한 자연어 처리 태스크에서 뛰어난 성능을 보여주었습니다.", 72),   # 72시간 전
    ("2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. "
    "GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 "
    "시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.", 8),    # 8시간 전
    ("2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. "
    "Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 "
    "클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.", 0),                    # 현재
]

for content, hours_ago in docs_with_time:
    time.sleep(0.1)
    doc = Document(
        page_content=content,
        metadata={"last_accessed_at": datetime.now() - timedelta(hours=hours_ago)}
    )
    time_weighted_retriever.add_documents([doc])
# TimeWeightedVectorStoreRetriever 생성
# decay_rate: 시간 감소율 — 매우 작은 값이면 사실상 시간순 정렬

# 시간차를 두고 문서 추가 (last_accessed_at으로 최신성 지정)


In [4]:
# 검색
query = "인공지능 기술 혁신"
docs = time_weighted_retriever.invoke(query)
# 아래에 코드를 작성하세요

for i, doc in enumerate(docs, 1):
    print(f"[{i}] {doc.page_content[:60]}...")


[1] 2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다....
[2] 2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4...
[3] 2020년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 OpenAI의 GPT-3 발표였습니다. 1...


/Users/elixir/dev/lecture/hk-toss/09_langchain/.venv/lib/python3.11/site-packages/langchain/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4a299b34-f982-4a27-bb23-6dd7603dac75', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 2, 11, 15, 430449), 'created_at': datetime.datetime(2026, 4, 10, 10, 11, 15, 430494), 'buffer_idx': 1}, page_content='2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.'), 0.3494241966652355), (Document(id='34418594-348d-4ce1-ac2a-f46fd69902dd', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 10, 11, 15, 643561), 'created_at': datetime.datetime(2026, 4, 10, 10, 11, 15, 643603), 'buffer_idx': 2}, page_content='2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.'), 0.317602735983631

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 최종 점수 = 의미 유사도 × (1 - decay_rate)^경과시간(시간 단위) |
| `decay_rate` 역할 | 값이 클수록 오래된 문서 점수가 빠르게 낮아짐 |
| 문서 접근 갱신 | 검색될 때마다 `last_accessed_at` 자동 갱신 |
| 적합한 용도 | 뉴스, 실시간 데이터, 날짜 민감한 FAQ |
| 주의 | 날짜 무관한 정보엔 비적합 — 유사도만 순수하게 쓰는 게 나을 수 있음 |

### `decay_rate` 선택 가이드

| `decay_rate` | 반감 속도 | 적합한 상황 |
|--------------|-----------|-------------|
| 0.01 | 약 70일 | 월간 업데이트 문서 |
| 0.1 | 약 7일 | 주간 뉴스, 블로그 |
| 0.5 | 약 1.4일 | 실시간 이슈 Q&A |
| 0.99 | 거의 즉시 | 당일 데이터만 유효한 경우 |

### 다음 노트북 예고

**10-Kiwi-BM25Retriever** — 한국어 형태소 분석기 Kiwi를 BM25와 결합해 한국어 키워드 검색 정확도를 높이는 방법을 배워요.